**CELL 1: Kết nối Google Drive & Cài đặt Thư viện**

In [ ]:
# ==========================================
# 1. KẾT NỐI GOOGLE DRIVE & CÀI ĐẶT THƯ VIỆN
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

# Cài thư viện (chỉ chạy lần đầu, ẩn thông báo cài đặt bằng -q)
!pip install pandas numpy pandas-ta-classic lightgbm xgboost scikit-learn matplotlib -q

import pandas as pd
import numpy as np
import pandas_ta_classic as ta
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import os

print("✅ Đã mount Google Drive và import thư viện thành công!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Đã mount Google Drive và import thư viện thành công!


**CELL 2: Đọc dữ liệu từ thư mục Google Drive**

In [ ]:
# ==========================================
# 2. ĐỌC DỮ LIỆU TỪ FOLDER DATA
# ==========================================
data_folder = '/content/drive/MyDrive/forex_data'   # Thay đổi nếu folder của bạn khác

def load_and_clean(filename):
    filepath = os.path.join(data_folder, filename)
    if not os.path.exists(filepath):
        print(f"❌ Không tìm thấy file: {filename} tại {data_folder}")
        return None
    df = pd.read_csv(filepath, parse_dates=['time'])
    df = df.sort_values('time').reset_index(drop=True)
    print(f"✅ Đã tải {filename}: {len(df):,} nến | Từ {df['time'].iloc[0].date()} đến {df['time'].iloc[-1].date()}")
    return df

print("Đang tải dữ liệu từ Google Drive...")
df_m5  = load_and_clean('XAUUSD_m5.csv')  # Cập nhật theo tên file thực tế của bạn
df_m15 = load_and_clean('XAUUSD_m15.csv')
df_h1  = load_and_clean('XAUUSD_h1.csv')
df_h4  = load_and_clean('XAUUSD_h4.csv')

# # ==========================================
# # ==========================================
# # 2. ĐỌC TỪ 1 FILE M5 VÀ TỰ RESAMPLE SANG ĐA KHUNG (NẾU CHỈ CÓ 1 FILE)
# # ==========================================
# data_folder = '/content/drive/MyDrive/forex_data'
# filepath = os.path.join(data_folder, 'XAUUSD_m5.csv')

# # 1. Đọc file M5 gốc
# df_m5 = pd.read_csv(filepath, parse_dates=['time'])
# df_m5 = df_m5.sort_values('time').reset_index(drop=True)
# print(f"✅ Đã tải file gốc M5: {len(df_m5):,} nến.")

# # Tạo bản sao có index là time để chuẩn bị gộp nến
# df_resample = df_m5.set_index('time')

# # 2. Tự động tạo khung M15 từ M5
# df_m15 = df_resample.resample('15Min').agg({
#     'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last', 'volume': 'sum', 'spread': 'mean'
# }).dropna().reset_index()

# # 3. Tự động tạo khung H1 từ M5
# df_h1 = df_resample.resample('1H').agg({
#     'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last', 'volume': 'sum', 'spread': 'mean'
# }).dropna().reset_index()

# # 4. Tự động tạo khung H4 từ M5
# df_h4 = df_resample.resample('4H').agg({
#     'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last', 'volume': 'sum', 'spread': 'mean'
# }).dropna().reset_index()

# print("✅ Đã tự động tạo các khung thời gian M15, H1, H4 chuẩn từ file M5 gốc!")

Đang tải dữ liệu từ Google Drive...
✅ Đã tải XAUUSD_m5.csv: 631,652 nến | Từ 2017-09-13 đến 2026-07-09
✅ Đã tải XAUUSD_m15.csv: 210,884 nến | Từ 2017-09-15 đến 2026-07-09
✅ Đã tải XAUUSD_h1.csv: 52,896 nến | Từ 2017-09-19 đến 2026-07-09
✅ Đã tải XAUUSD_h4.csv: 13,349 nến | Từ 2018-04-25 đến 2026-07-09


**CELL 3: Feature Engineering (Đa khung thời gian & Chuẩn hóa)**

In [ ]:
# ==========================================
# 3. FEATURE ENGINEERING VỚI GIÁ TƯƠNG ĐỐI (STATIONARY FEATURES)
# ==========================================
print("Đang tính toán các đặc trưng tương đối (Loại bỏ giá tuyệt đối)...")

# --- TÍNH TOÁN TRÊN KHUNG M5 (Khung vào lệnh gốc) ---
df_m5.ta.ema(length=20, append=True)
df_m5.ta.rsi(length=14, append=True)
df_m5.ta.atr(length=14, append=True)
df_m5.ta.bbands(length=20, std=2, append=True)

# Chuyển đổi các đặc trưng M5 sang dạng STATIONARY (Tương đối)
df_m5['ret_close'] = df_m5['close'].pct_change()
df_m5['pct_body'] = (df_m5['close'] - df_m5['open']) / df_m5['open']
df_m5['atr_norm_upper_shadow'] = (df_m5['high'] - df_m5[['open', 'close']].max(axis=1)) / df_m5['ATRr_14']
df_m5['atr_norm_lower_shadow'] = (df_m5[['open', 'close']].min(axis=1) - df_m5['low']) / df_m5['ATRr_14']
df_m5['dist_EMA20_pct'] = (df_m5['close'] - df_m5['EMA_20']) / df_m5['EMA_20']
df_m5['dist_BBU_pct'] = (df_m5['BBU_20_2.0'] - df_m5['close']) / df_m5['close']
df_m5['dist_BBL_pct'] = (df_m5['close'] - df_m5['BBL_20_2.0']) / df_m5['close']

# --- TÍNH TOÁN TRÊN KHUNG M15 ---
df_m15.ta.ema(length=20, append=True)
df_m15.ta.macd(append=True)
df_m15['dist_EMA20_M15_pct'] = (df_m15['close'] - df_m15['EMA_20']) / df_m15['EMA_20']
df_m15['macd_norm'] = df_m15['MACD_12_26_9'] / df_m15['close']
df_m15_features = df_m15[['time', 'dist_EMA20_M15_pct', 'macd_norm']]

# --- TÍNH TOÁN TRÊN KHUNG H1 ---
df_h1.ta.ema(length=20, append=True)
df_h1.ta.rsi(length=14, append=True)
df_h1['dist_EMA20_H1_pct'] = (df_h1['close'] - df_h1['EMA_20']) / df_h1['EMA_20']
df_h1_features = df_h1[['time', 'dist_EMA20_H1_pct', 'RSI_14']]
df_h1_features = df_h1_features.rename(columns={'RSI_14': 'RSI_14_H1'})

# --- TÍNH TOÁN TRÊN KHUNG H4 ---
df_h4.ta.ema(length=20, append=True)
df_h4.ta.rsi(length=14, append=True)
df_h4['dist_EMA20_H4_pct'] = (df_h4['close'] - df_h4['EMA_20']) / df_h4['EMA_20']
df_h4_features = df_h4[['time', 'dist_EMA20_H4_pct', 'RSI_14']]
df_h4_features = df_h4_features.rename(columns={'RSI_14': 'RSI_14_H4'})

# --- GỘP ĐA KHUNG THỜI GIAN (Dùng merge_asof để tránh look-ahead bias) ---
merged_df = pd.merge_asof(df_m5, df_m15_features, on='time', direction='backward')
merged_df = pd.merge_asof(merged_df, df_h1_features, on='time', direction='backward')
merged_df = pd.merge_asof(merged_df, df_h4_features, on='time', direction='backward')

print("✅ Đã tạo xong đặc trưng đa khung thời gian!")

**CELL 4: Gán nhãn Triple-Barrier Method**

In [ ]:
# ==========================================
# 4. GÁN NHÃN TRIPLE-BARRIER METHOD (DỰA TRÊN SL/TP THỰC TẾ)
# ==========================================
print("Đang gán nhãn dữ liệu bằng thuật toán Triple-Barrier...")

def apply_triple_barrier(df, pt_sl=[3.0, 1.5], max_holding_candles=12):
    high_prices  = df['high'].values
    low_prices   = df['low'].values
    close_prices = df['close'].values
    atrs         = df['ATRr_14'].values

    n = len(df)
    labels = np.full(n, np.nan)

    for i in range(n - max_holding_candles):
        if np.isnan(atrs[i]):
            continue

        entry_price = close_prices[i]
        atr_now = atrs[i]

        tp_barrier = entry_price + pt_sl[0] * atr_now
        sl_barrier = entry_price - pt_sl[1] * atr_now

        label_i = 0.0  # Mặc định đi ngang (Sideway)
        for j in range(1, max_holding_candles + 1):
            curr_high = high_prices[i + j]
            curr_low  = low_prices[i + j]

            if curr_low <= sl_barrier and curr_high >= tp_barrier:
                label_i = -1.0 # Quét cả 2 râu -> Ưu tiên rủi ro dính SL trước
                break
            elif curr_low <= sl_barrier:
                label_i = -1.0
                break
            elif curr_high >= tp_barrier:
                label_i = 1.0
                break

        labels[i] = label_i

    df['target'] = labels
    return df

# Gọi hàm gán nhãn
merged_df = apply_triple_barrier(merged_df, pt_sl=[3.0, 2.0], max_holding_candles=12)
merged_df.dropna(inplace=True)

# Kiểm tra tỷ lệ phân bố các nhãn
print("\n📊 Tỷ lệ phân bố nhãn target:")
print(merged_df['target'].value_counts(normalize=True))

**CELL 5: Chia tập dữ liệu (Train / Val / Test) độc lập**

In [ ]:
# ==========================================
# 5. ĐỊNH NGHĨA FEATURES VÀ CHIA TẬP DATA 3 PHẦN ĐỘC LẬP
# ==========================================
# Danh sách feature loại bỏ hoàn toàn giá tuyệt đối để tránh Overfitting
feature_cols = [
    'volume', 'spread', 'RSI_14',
    'ret_close', 'pct_body', 'atr_norm_upper_shadow', 'atr_norm_lower_shadow',
    'dist_EMA20_pct', 'dist_BBU_pct', 'dist_BBL_pct',
    'dist_EMA20_M15_pct', 'macd_norm',
    'dist_EMA20_H1_pct', 'RSI_14_H1',
    'dist_EMA20_H4_pct', 'RSI_14_H4'
]

# Set index theo thời gian trước khi chia tập
if 'time' in merged_df.columns:
    merged_df.set_index('time', inplace=True)

# Tách dữ liệu theo dòng thời gian nghiêm ngặt
train_data = merged_df.loc[:'2025-12-30']
val_data   = merged_df.loc['2026-05-01':'2026-06-30']
test_data  = merged_df.loc['2026-07-01':]

X_train, y_train = train_data[feature_cols], train_data['target'].astype(int)
X_val, y_val     = val_data[feature_cols], val_data['target'].astype(int)
X_test, y_test   = test_data[feature_cols], test_data['target'].astype(int)

print(f"📊 Kích thước các tập dữ liệu:")
print(f"   ↳ Mẫu tập Train (Quá khứ -> '2025-12-30'): {len(X_train):,}")
print(f"   ↳ Mẫu tập Validation ('2026-05-01':'2026-06-30'):       {len(X_val):,}")
print(f"   ↳ Mẫu tập Test (2026-07-01 -> Nay):      {len(X_test):,}")
print(f"   ↳Done")

**CELL 6: Huấn luyện mô hình LightGBM**

In [ ]:
# ==========================================
# 6. HUẤN LUYỆN LIGHTGBM (ĐÃ FIX HOÀN TOÀN)
# ==========================================
print("=== HUẤN LUYỆN LIGHTGBM (Sử dụng tập Validation riêng biệt) ===")

# Ép kiểu integer
y_train = y_train.astype(int)
y_val   = y_val.astype(int)
y_test  = y_test.astype(int)

# Kiểm tra nhãn thực tế có trong data
print("Nhãn trong Train:", y_train.value_counts().sort_index())
print("Nhãn trong Val:", y_val.value_counts().sort_index())

# Lấy classes thực tế có trong data
actual_classes = np.unique(y_train)
print("Classes thực tế:", actual_classes)

# TÍNH CLASS WEIGHT AN TOÀN
# ==========================================
from sklearn.utils.class_weight import compute_class_weight

# Lấy classes thực tế có trong dữ liệu
actual_classes = np.sort(y_train.unique().astype(int))
print("Classes thực tế:", actual_classes)

# Tính class weight
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=actual_classes,
    y=y_train.astype(int)
)

class_weight_dict = dict(zip(actual_classes, class_weights))
print("Class weights:", class_weight_dict)
class_weight_dict = dict(zip(actual_classes, class_weights))

print("Class weights:", class_weight_dict)

model_lgb = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.02,
    max_depth=6,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    class_weight=class_weight_dict
)

model_lgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=True)]
)

print("✅ Hoàn thành huấn luyện LightGBM!")

In [ ]:
# ==========================================
# 7. HUẤN LUYỆN XGBOOST (AN TOÀN CHO EVAL_SET)
# ==========================================
print("=== HUẤN LUYỆN XGBOOST (Sử dụng tập Validation riêng biệt) ===")

# Đổi nhãn từ [-1, 0, 1] thành [0, 1, 2] do quy định nhãn đa lớp của XGBoost
y_train_xgb = y_train + 1
y_val_xgb   = y_val + 1
y_test_xgb  = y_test + 1

# Tính toán class_weight thủ công cho XGBoost vì không hỗ trợ trực tiếp từ 'balanced'
from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_xgb)

model_xgb = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.02,
    max_depth=6,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=30,
    eval_metric="mlogloss"
)

model_xgb.fit(
    X_train, y_train_xgb,
    sample_weight=sample_weights,
    eval_set=[(X_val, y_val_xgb)],
    verbose=True
)
print("✅ Hoàn thành huấn luyện XGBoost!")

In [ ]:
# ==========================================
# 8. ĐÁNH GIÁ THỰC TẾ TRÊN TẬP TEST BÍ MẬT & LƯU MODEL
# ==========================================
print("\n" + "="*60)
print("BÁO CÁO ĐÁNH GIÁ CHUẨN TRÊN TẬP TEST BÍ MẬT (2026 - NAY)")
print("="*60)

# 1. Đánh giá LightGBM
y_pred_lgb = model_lgb.predict(X_test)
print("\n[LIGHTGBM EVALUATION] - Nhãn: -1 (Thua/Chạm SL), 0 (Sideway), 1 (Thắng/Chạm TP):")
print(classification_report(y_test, y_pred_lgb))

# 2. Đánh giá XGBoost
y_pred_xgb = model_xgb.predict(X_test) - 1 # Đổi ngược từ [0,1,2] về lại [-1,0,1]
print("\n[XGBOOST EVALUATION] - Nhãn: -1 (Thua/Chạm SL), 0 (Sideway), 1 (Thắng/Chạm TP):")
print(classification_report(y_test, y_pred_xgb))

# 3. Tiến hành tạo thư mục và lưu trữ Model vào Google Drive
model_folder = '/content/drive/MyDrive/models'
os.makedirs(model_folder, exist_ok=True)

# Lưu file mô hình
model_lgb.booster_.save_model(os.path.join(model_folder, 'lgb_gold_v1.txt'))
model_xgb.save_model(os.path.join(model_folder, 'xgb_gold_v1.json'))

print(f"\n✅ Model đã được xuất và lưu thành công vào Google Drive tại: {model_folder}")